In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

Installing Libraries
Installing Required Libraries

In [ ]:
#install dependencies
!pip -q install \
langchain \
langchain-community \
langchain-groq \
sentence-transformers \
faiss-cpu \
youtube-transcript-api \
langchain-text-splitters

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.2.43 which is incompatible.
langgraph 1.2.5 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.


In [13]:
from youtube_transcript_api import YouTubeTranscriptApi

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

STEP-1a : DOCUMENT INGESTION


In [14]:
video_id = "5sLYAQS9sWQ"

ytt = YouTubeTranscriptApi()

transcript_list = ytt.fetch(
    video_id,
    languages=["en-US", "en"]
)

transcript = " ".join(
    chunk.text for chunk in transcript_list
)

print(transcript[:1000])

GPT, or Generative Pre-trained Transformer, is a large language model, or an LLM, that can generate human-like text. And I've been using GPT in its various forms for years. In this video we are going to number 1, ask "what is an LLM?" Number 2, we are going to describe how they work. And then number 3, we're going to ask, "what are the business applications of LLMs?" So let's start with number 1, "what is a large language model?" Well, a large language model is an instance of something else called a foundation model. Now foundation models are pre-trained on large amounts of unlabeled and self-supervised data, meaning the model learns from patterns in the data in a way that produces generalizable and adaptable output. And large language models are instances of foundation models applied specifically to text and text-like things. I'm talking about things like code. Now, large language models are trained on large datasets of text, such as books, articles and conversations. And look, when w

STEP-1b: TEXT SPLITTING


In [18]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.create_documents([transcript])

print(len(chunks))

chunks[5]

11


Document(page_content='amounts of text data that goes into these things. As for the architecture, this is a neural network and for GPT that is a transformer. And the transformer architecture enables the model to handle sequences of data like sentences or lines of code. And transformers are designed to understand the context of each word in a sentence by considering it in relation to every other word. This allows the model to build a comprehensive understanding of the sentence structure and the meaning of the words')

STEP-1c : Embedding Gen and store in vector

In [10]:
!pip uninstall -y sentence-transformers transformers
!pip install -q sentence-transformers==2.7.0 transformers==4.41.2

Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2


In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_6659/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
#Storing in FAISS
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)
vector_store.index_to_docstore_id

{0: '3b3a787e-a861-43a4-b6d1-44ac9bc077da',
 1: 'fb7e7871-b32d-48b2-ab96-2f99cafea668',
 2: '19a70d54-fb41-4ca4-bb5b-1c1602ecc263',
 3: '52644dde-badb-43ef-b005-381aeb9ebef9',
 4: '1dbaa7fb-fdc3-4202-8258-0e124f9e8ab3',
 5: '7ca0a9ce-3a0a-4d72-adaa-cace78b97f99',
 6: 'f08ac2c4-eeac-4f77-8205-671cdb5206d8',
 7: '472193a9-85f2-4cb9-9b7a-a7a0a0e5cf2d',
 8: '3fa85967-0962-4dc5-8ea2-7800766b4606',
 9: '5b8a04e6-4016-4538-a160-e4aacfc6b929',
 10: '4a08f149-e227-45da-a556-0766f0cd7b32'}

In [27]:
doc_id = "19a70d54-fb41-4ca4-bb5b-1c1602ecc263"

document = vector_store.docstore.search(doc_id)

print(document)

page_content='specifically to text and text-like things. I'm talking about things like code. Now, large language models are trained on large datasets of text, such as books, articles and conversations. And look, when we say "large", these models can be tens of gigabytes in size and trained on enormous amounts of text data. We're talking potentially petabytes of data here. So to put that into perspective, a text file that is, let's say, one gigabyte in size, that can store about 178 million words. A lot of'


STEP-2 : Retrieval

In [29]:
query = "Explain what is GPT"

results = vector_store.similarity_search(
    query,
    k=3
)

In [31]:
print(results)

[Document(page_content='GPT, or Generative Pre-trained Transformer, is a large language model, or an LLM, that can generate human-like text. And I\'ve been using GPT in its various forms for years. In this video we are going to number 1, ask "what is an LLM?" Number 2, we are going to describe how they work. And then number 3, we\'re going to ask, "what are the business applications of LLMs?" So let\'s start with number 1, "what is a large language model?" Well, a large language model is an instance of something else'), Document(page_content="has, the more complex it can be. GPT-3, for example, is pre-trained on a corpus of actually 45 terabytes of data, and it uses 175 billion ML parameters. All right, so how do they work? Well, we can think of it like this. LLM equals three things: data, architecture, and lastly, we can think of it as training. Those three things are really the components of an LLM. Now, we've already discussed the enormous amounts of text data that goes into these t

In [50]:
#Instead of manually searching every time
retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)
docs = retriever.invoke("Explain GPT")
print(docs)

[Document(page_content='GPT, or Generative Pre-trained Transformer, is a large language model, or an LLM, that can generate human-like text. And I\'ve been using GPT in its various forms for years. In this video we are going to number 1, ask "what is an LLM?" Number 2, we are going to describe how they work. And then number 3, we\'re going to ask, "what are the business applications of LLMs?" So let\'s start with number 1, "what is a large language model?" Well, a large language model is an instance of something else'), Document(page_content="has, the more complex it can be. GPT-3, for example, is pre-trained on a corpus of actually 45 terabytes of data, and it uses 175 billion ML parameters. All right, so how do they work? Well, we can think of it like this. LLM equals three things: data, architecture, and lastly, we can think of it as training. Those three things are really the components of an LLM. Now, we've already discussed the enormous amounts of text data that goes into these t

STEP-3 : Augmentation


In [51]:
#Load Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0   #temperature = 0 means Always give the most deterministic answer.
)

In [52]:
#Prompt
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Answer ONLY from the given context.

Context:
{context}

Question:
{question}
""",
    input_variables=["context","question"]
)

In [53]:
#Combine Retrieved Chunks
context = "\n\n".join(
    doc.page_content
    for doc in docs
)

In [54]:
#Create Final Prompt
final_prompt = prompt.invoke(
    {
        "context": context,
        "question": "Explain GPT"
    }
)

Prompt

+
Retrieved Context

+
Question

STEP-4 : GENERATION

In [55]:
#Generate Answer
answer = llm.invoke(final_prompt)

print(answer.content)

GPT, or Generative Pre-trained Transformer, is a large language model (LLM) that can generate human-like text. It is pre-trained on a large corpus of data, such as 45 terabytes of data, and uses a large number of machine learning parameters, like 175 billion. It uses a transformer architecture, which is a type of neural network that enables the model to handle sequences of data and understand the context of each word in a sentence.


STEP-5 : BUILDING A CHAIN

In [56]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda,
    RunnablePassthrough
)

from langchain_core.output_parsers import StrOutputParser

In [57]:
#Format the retrieved documents
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [58]:
#Parallel chain
parallel_chain = RunnableParallel(
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
)  #here output of retriever becomes input for format_docs function

When you ask

"What is GPT?"

it automatically does:

        Question
          │
      ├───────────────┐
      │               │
      ▼               ▼
    Retriever        Original Question
      │               │
      ▼               │
    Format Docs       │
      │               │
      └──────┬────────┘
             ▼
     {context, question}

In [59]:
#Final chain
parser = StrOutputParser()

main_chain = ( parallel_chain | prompt | llm | parser )

In [60]:
main_chain.invoke(
    "What is GPT?"
)

main_chain.invoke(
    "Who is Demis Hassabis?"
)

main_chain.invoke(
    "Did they discuss nuclear fusion?"
)

'No, they did not discuss nuclear fusion. The context only talks about large language models (LLMs) and their components, such as data, architecture, and training.'

    User Question
       │
       ▼
    Retriever (FAISS)
       │
       ▼
    Top 3 Relevant Chunks
       │
       ▼
    Format Docs
       │
       ▼
    Prompt Template
       │
       ▼
    Groq (Llama 3.3 70B)
       │
      ▼
    Output Parser
       │
      ▼
    Final Answer